In [7]:
import numpy as np
from scipy.special import roots_hermitenorm, eval_hermitenorm, factorial
from scipy.optimize import brentq
import pandas as pd

def gaussian_expectation(f, n_quad=200):
    """
    Compute E[f(Z)] for Z ~ N(0,1)
    """
    x, w = roots_hermitenorm(n_quad)
    return np.sum(w * f(x)) / np.sqrt(2 * np.pi)

def normalize_transform(f, n_quad=200):
    """
    Return a centered variance-1 version of f
    """

    mean = gaussian_expectation(f, n_quad)
    var = gaussian_expectation(
        lambda z: (f(z)-mean)**2,
        n_quad
    )
    sd = np.sqrt(var)

    return lambda z: (f(z)-mean)/sd

def get_hermite_coeff(f, k, n_quad=200):
    """
    Orthonormal Hermite coefficient

    c_k = E[f(Z)He_k(Z)] / sqrt(k!)
    where Z ~ N(0, 1)
    """
    x, w = roots_hermitenorm(n_quad)

    expectation = (
        np.sum(w * f(x) * eval_hermitenorm(k, x))
        / np.sqrt(2 * np.pi)
    )

    return expectation / np.sqrt(factorial(k, exact=False))

def hermite_distribution(f, max_order=20, n_quad=150):
    """
    Calculate Hermite coefficients and energies for a
    centered, variance-normalized transform.

    Returns
    dict containing:
        coefficients
        energies
        nonlinear_energy
        first_order_energy
        captured_energy
    """

    g = normalize_transform(f, n_quad)

    coeffs = np.array([
        get_hermite_coeff(g, k, n_quad)
        for k in range(max_order + 1)
    ])

    energies = coeffs**2

    return {
        "coefficients": coeffs,
        "energies": energies,
        "first_order_energy": energies[1],
        "nonlinear_energy": 1 - energies[1],
        "captured_energy": np.sum(energies)
    }

def find_strength(
    transform,
    target_nonlinear_energy,
    strength_bounds,
    max_order=20,
    n_quad=150
):
    """
    Find theta such that

        1 - rho^2 = target_nonlinear_energy

    where rho = Corr(Z, T_theta(Z)).

    Parameters
    ----------
    transform:
        Function of form transform(z, strength)

    target_nonlinear_energy:
        Desired value of 1 - rho^2

    strength_bounds:
        Tuple (low, high) in which to search

    max_order:
        Maximum Hermite order returned

    Returns
    -------
    result : dict
        strength
        nonlinear_energy
        first_order_energy
        coefficients
        energies
        captured_energy
    """

    def objective(strength):

        f = lambda z: transform(z, strength)

        result = hermite_distribution(
            f,
            max_order=max_order,
            n_quad=n_quad
        )

        return (
            result["nonlinear_energy"]
            - target_nonlinear_energy
        )

    strength = brentq(
        objective,
        strength_bounds[0],
        strength_bounds[1]
    )

    f = lambda z: transform(z, strength)

    result = hermite_distribution(
        f,
        max_order=max_order,
        n_quad=n_quad
    )

    result["strength"] = strength

    return result

def cubic(z, strength):
    return z + strength * z**3


def quintic(z, strength):
    return z + strength * z**5


def exponential(z, strength):
    if strength == 0:
        return z
    return np.expm1(strength * z) / strength


def quad_cubic(z, a):
    z = np.asarray(z, dtype=float)
    return z + a * z**2 + (a**2 / 3.0) * z**3


for transform in [cubic, quintic, exponential, quad_cubic]:     
    print(transform)  
    for target in [0.05, 0.1, 0.15, 0.2, 0.3]:
            
        result = find_strength(
            transform=transform,
            target_nonlinear_energy=target,
            strength_bounds=(0.0, 5),
            max_order=15
        )

        print("Required strength:", result["strength"])

def get_hermite_coeffs(transform_fn, strength, k_max=5, n_quad=200):
    x, w = roots_hermitenorm(n_quad)
    w = w / np.sqrt(2 * np.pi)

    y = transform_fn(x, strength)

    y_mean = np.sum(w * y)
    y_sd = np.sqrt(np.sum(w * (y - y_mean)**2))
    y = (y - y_mean) / y_sd

    coeffs = []

    for k in range(k_max + 1):
        c_k = np.sum(
            w * y * eval_hermitenorm(k, x)
        ) / np.sqrt(factorial(k))

        coeffs.append(c_k)

    return np.array(coeffs)

def sinh(z, a):
    if np.isclose(a, 0.0):
        return z
    return np.sinh(a * z) / a


def tanh(z, a):
    return z + a * np.tanh(z)

TRANSFORMS = {
    "cubic": cubic,
    "sinh": sinh,
    "exp": exponential,
    "tanh": tanh,
}


STRENGTHS = {
    "cubic": [0.0, 0.10, 0.25, 0.50, 1.00],
    "sinh":  [0.0, 0.35, 0.70, 1.00, 1.30],
    "exp":   [0.0, 0.20, 0.40, 0.60, 0.80],
    "tanh":  [0.0, 0.25, 0.50, 1.00, 2.00],
}

hermite_rows = []

for transform_name, strengths in STRENGTHS.items():
    for strength in strengths:

        coeffs = get_hermite_coeffs(
            TRANSFORMS[transform_name],
            strength
        )

        row = {
            "transform": transform_name,
            "strength": strength
        }

        for k, c in enumerate(coeffs):
            row[f"c{k}"] = c
            row[f"energy{k}"] = c**2

        row["higher_order_energy"] = np.sum(coeffs[2:]**2)

        row["weighted_higher_order_energy"] = np.sum(
            np.arange(2, len(coeffs)) * coeffs[2:]**2
        )

        hermite_rows.append(row)


hermite_results = pd.DataFrame(hermite_rows)

print(hermite_results)

hermite_results.to_csv(
    "../results/hermite_metrics.csv",
    index=False
)

<function cubic at 0x000001D6C056B880>
Required strength: 0.13025788811458458
Required strength: 0.22996598285207484
Required strength: 0.3532380757938138
Required strength: 0.5265986323710918
Required strength: 1.3483314773547856
<function quintic at 0x000001D6C0654540>
Required strength: 0.009807620374411513
Required strength: 0.015267537520168045
Required strength: 0.02046040925828413
Required strength: 0.02586276137841687
Required strength: 0.038479589734596774
<function exponential at 0x000001D6E26244A0>
Required strength: 0.3189425386491181
Required strength: 0.45513350013398407
Required strength: 0.5627497349482242
Required strength: 0.656385717230547
Required strength: 0.8218707884650162
<function quad_cubic at 0x000001D6C1C53D80>
Required strength: 0.16592806829167261
Required strength: 0.2476383907709284
Required strength: 0.3223790465869065
Required strength: 0.3994894169143198
Required strength: 0.591223660701687
   transform  strength            c0       energy0        c1 